# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv(r'C:\Users\karna\Downloads\world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** A line chart is appropriate because population is observed over time and the question asks for a trend. I index every continent to 100 in 1970 so growth is comparable despite different starting sizes; grey context lines and a blue highlight make Africa's leading growth immediately visible (Lecture 3).

**What the chart shows:** Looking at the chart, Africa stands out because its population increased much faster than the other continents. It grew by about 290% between 1970 and 2022, while Europe grew by only around 13%. I used a line chart because it makes changes over time easy to follow. Setting 1970 to 100 for every continent gives a fair comparison, even though their starting populations were different.


In [2]:
# The key finding is shown through the title, labels and annotations.
# Q1 — Lecture 3: line chart, index to 1970, grey context + accent
continent_year = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum())
base = continent_year.loc[continent_year['Year'] == 1970, ['Continent','Population']].rename(columns={'Population':'Base'})
continent_year = continent_year.merge(base, on='Continent')
continent_year['Index'] = 100 * continent_year['Population'] / continent_year['Base']
order = ['Europe','North America','Asia','South America','Oceania','Africa']
fig = px.line(continent_year, x='Year', y='Index', color='Continent',
              category_orders={'Continent':order},
              color_discrete_map={c:('#087E8B' if c=='Africa' else '#B5C4D1') for c in order},
              markers=True, template='plotly_white',
              title='Africa grew fastest: its population nearly quadrupled since 1970',
              labels={'Index':'Population index (1970 = 100)'})
fig.update_traces(line_width=2, marker_size=5)
fig.update_traces(line_width=4, marker_size=8, selector=dict(name='Africa'))
for c in order:
    end = continent_year[(continent_year.Continent==c)&(continent_year.Year==2022)].iloc[0]
    fig.add_annotation(x=2022, y=end.Index, text=f'{c} {end.Index:.0f}', showarrow=False,
                       xanchor='left', xshift=9, font=dict(size=11,color='#087E8B' if c=='Africa' else '#64748B'))
fig.update_layout(showlegend=False, height=520, margin=dict(l=65,r=170,t=90,b=60),
                  title_x=0.02, xaxis_title='', yaxis_title='Index (1970 = 100)')
fig.update_xaxes(showgrid=False, tickvals=sorted(continent_year.Year.unique()))
fig.update_yaxes(showgrid=True, gridcolor='#E9EEF3', rangemode='tozero')



# The answer is printed on the figure, not hidden in the hover tooltip.
final = continent_year[continent_year.Year.eq(2022)].sort_values('Index',ascending=False)
winner=final.iloc[0]
fig.add_annotation(x=.02,y=1.12,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>{winner.Continent} grew fastest (+{winner.Index-100:.0f}% since 1970)</b>',
                   font=dict(size=15,color='#087E8B'))
fig.update_layout(margin=dict(l=65,r=180,t=135,b=60))
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** A sorted horizontal bar chart makes ranking and country names easy to read. Bars start at zero, while orange highlights African countries against grey context bars; direct values and a count annotation answer both parts of the question (Lecture 2).

**What the chart shows:** India added the most people between 1970 and 2022, about 860 million, followed by China with about 603 million. Four of the top 15 countries are in Africa: Nigeria, Ethiopia, DR Congo and Egypt. I chose horizontal bars because country names are easier to read and the ranking is immediately clear. Highlighting African countries in orange helps answer the second part without making the chart too colourful.


In [3]:
# The key finding is shown through the title, labels and annotations.
# Q2 — Lecture 2: sorted horizontal bars, zero baseline, Africa accent
ranked = df.assign(Added=df['2022 Population']-df['1970 Population']).nlargest(15,'Added').copy()
ranked['Added_m'] = ranked['Added']/1e6
ranked = ranked.sort_values('Added_m')
ranked['Group'] = np.where(ranked.Continent.eq('Africa'),'Africa','Other continents')
africa_count = int(ranked.Continent.eq('Africa').sum())
fig = px.bar(ranked, x='Added_m', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa':'#7856A6','Other continents':'#B5C4D1'},
             text='Added_m', template='plotly_white',
             title=f'{africa_count} African countries rank among the 15 largest population gains',
             labels={'Added_m':'People added, 1970–2022 (millions)'})
fig.update_traces(texttemplate='%{x:,.0f}m', textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, height=620, title_x=.02, margin=dict(l=120,r=95,t=85,b=60),
                  yaxis_title='', xaxis_title='Population added (millions)', bargap=.3)
fig.update_xaxes(range=[0,ranked.Added_m.max()*1.17],showgrid=False,zeroline=False)
fig.update_yaxes(showgrid=False)



fig.add_annotation(x=.02,y=1.11,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>India ranks #1; {africa_count} of the top 15 are African (orange).</b>',
                   font=dict(size=14,color='#65418E'))
fig.update_layout(margin=dict(l=120,r=100,t=135,b=60))
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** A scatter plot shows the relationship between two numerical variables and reveals exceptions. Logarithmic axes accommodate large differences in country size, while continent colours and labelled outliers reveal patterns without treating area as a cause of population (Lecture 4).

**What the chart shows:** There is some relationship between land area and population, but it is not a strong rule that bigger countries always have more people. Russia and Australia have huge land areas, while Bangladesh has a much smaller area but a large population. The scatter plot lets me compare both numerical variables and spot these exceptions. I used logarithmic axes because the countries vary enormously in size and population.


In [4]:
# The key finding is shown through the title, labels and annotations.
# Q3 — Lecture 4: scatter, two continuous variables, log scales and labelled exceptions
scatter = df[(df.Area>0)&(df['2022 Population']>0)].copy()
scatter['Population_m'] = scatter['2022 Population']/1e6
palette = dict(zip(sorted(scatter.Continent.unique()),['#087E8B','#7856A6','#D08B36','#527D9A','#C05D7C','#617B58'][:scatter.Continent.nunique()]))
fig = px.scatter(scatter, x='Area', y='Population_m', color='Continent',
                 hover_name='Country', color_discrete_map=palette, opacity=.75,
                 log_x=True, log_y=True, template='plotly_white',
                 title='Land area alone does not determine a country’s population',
                 labels={'Area':'Land area (km², log scale)','Population_m':'2022 population (millions, log scale)'})
fig.update_traces(marker=dict(size=8,line=dict(width=.4,color='white')))
for name in ['India','China','Russia','Bangladesh','Singapore','Australia']:
    t=scatter.loc[scatter.Country.eq(name)]
    if len(t):
        r=t.iloc[0]
        fig.add_annotation(x=r.Area,y=r.Population_m,text=name,showarrow=True,
                           ax=28,ay=-24,arrowwidth=.8,arrowcolor='#64748B',font=dict(size=10))
fig.update_layout(height=600,title_x=.02,legend_title_text='Continent',
                  legend=dict(orientation='h',y=-.2,x=0),margin=dict(l=70,r=50,t=90,b=110))
fig.update_xaxes(showgrid=True,gridcolor='#E9EEF3',zeroline=False)
fig.update_yaxes(showgrid=True,gridcolor='#E9EEF3',zeroline=False)



fig.add_annotation(x=.01,y=1.10,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text='<b>Country sizes and populations vary widely; area alone is not a reliable predictor.</b>',
                   font=dict(size=13,color='#24364B'))
fig.update_layout(margin=dict(l=75,r=50,t=135,b=110))
fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** Box plots show the median, interquartile range and outliers, which averages conceal. A logarithmic density axis keeps highly crowded territories from compressing the other distributions; the title and hover values explain what is compared (Lecture 5).

**What the chart shows:** Population density is very different across countries, even within the same continent. In this dataset, North America has the highest median country or territory density, at about 221 people per km², while South America has the lowest, at about 18. Some very densely populated places are far above the typical values. A box plot is useful here because it shows the median, spread and outliers instead of hiding the differences behind one average.


In [5]:
# The key finding is shown through the title, labels and annotations.
# Q4 — Lecture 5: compare distributions, medians, spread and outliers
medians = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
fig = px.box(df, x='Continent', y='Density', color='Continent', points='outliers',
             category_orders={'Continent':medians.index.tolist()},
             color_discrete_map={c:('#087E8B' if c==medians.index[0] else '#B5C4D1') for c in medians.index},
             log_y=True, template='plotly_white',
             title='Population density varies widely within every continent',
             labels={'Density':'People per km² (log scale)'},hover_name='Country')
fig.update_traces(boxmean=False,marker=dict(size=5,opacity=.6))
fig.update_layout(showlegend=False,height=550,title_x=.02,xaxis_title='',
                  yaxis_title='People per km² (log scale)',margin=dict(l=75,r=35,t=85,b=70))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True,gridcolor='#E9EEF3')



fig.add_annotation(x=.01,y=1.13,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>Highest median: {medians.index[0]} ({medians.iloc[0]:,.1f}/km²); lowest: {medians.index[-1]} ({medians.iloc[-1]:,.1f}/km²).</b>',
                   font=dict(size=13,color='#087E8B'))
fig.add_annotation(x=.01,y=1.07,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text='Boxes = middle 50% of countries; centre lines = medians; dots = outliers.',
                   font=dict(size=11,color='#64748B'))
fig.update_layout(margin=dict(l=75,r=35,t=145,b=70))
fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** A choropleth is appropriate because the location of population growth and decline is central to the question. A blue–orange diverging scale centred at zero separates increases from decreases, with hover labels showing exact percentage changes (Lecture 6).

**What the chart shows:** Most countries grew between 2000 and 2022, but 34 countries or territories had a population decline. I used a world map because the question is about where these changes happened. The two-sided colour scale separates decreases from increases, with zero as the midpoint. Hovering over a country gives the exact percentage, so the map stays readable without putting numbers everywhere.


In [6]:
# The key finding is shown through the title, labels and annotations.
# Q5 — Lecture 6: choropleth with zero-centred diverging colour
geo = df.copy()
geo['Growth_pct'] = 100*(geo['2022 Population']/geo['2000 Population']-1)
shrunk = int((geo.Growth_pct<0).sum())
limit = max(abs(geo.Growth_pct.min()),abs(geo.Growth_pct.max()))
fig = px.choropleth(geo, locations='CCA3', locationmode='ISO-3', color='Growth_pct',
                    hover_name='Country', hover_data={'Growth_pct':':.1f','CCA3':False},
                    color_continuous_scale=[(0,'#7856A6'),(.5,'#F7F9FC'),(1,'#087E8B')],
                    range_color=[-limit,limit], template='plotly_white',
                    title=f'{shrunk} countries/territories lost population between 2000 and 2022',
                    labels={'Growth_pct':'Change (%)'})
fig.update_geos(showframe=False,showcoastlines=False,projection_type='natural earth',
                showland=True,landcolor='#F0F0F0',showocean=True,oceancolor='white')
fig.update_layout(height=580,title_x=.02,margin=dict(l=10,r=10,t=85,b=20),
                  coloraxis_colorbar=dict(title='Change (%)',ticksuffix='%'))



# Fixed, interpretable colour range: prevents very high growth from washing out other countries.
# Orange = decline; white = zero; blue = growth. Exact values remain visible on hover.
fig.update_coloraxes(cmin=-100,cmax=100,cmid=0,
                     colorbar=dict(title='Population change',ticksuffix='%',tickvals=[-100,-50,0,50,100]))
fig.add_annotation(x=.02,y=1.11,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>{shrunk} countries/territories shrank (orange); blue countries grew.</b>',
                   font=dict(size=14,color='#24364B'))
fig.update_layout(margin=dict(l=10,r=10,t=135,b=20))
fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** A treemap represents additive population totals through rectangle area and shows the nested continent → country hierarchy in one chart. Muted continent colours, readable labels and detailed hover values emphasize relative magnitude without unnecessary decoration (Lecture 7).

**What the chart shows:** Asia has the largest share of the world's population in 2022, at roughly 59%, while Africa is next at about 18%. The treemap makes this easy to see because the size of each rectangle represents population. Grouping countries inside their continents also shows how the world total is divided at both levels. This is more useful for this question than a long list of countries.


In [7]:
# The key finding is shown through the title, labels and annotations.
# Q6 — Lecture 7: additive measure + nested hierarchy
pop = df.copy()
pop['Population_m'] = pop['2022 Population']/1e6
continent_colors = dict(zip(sorted(pop.Continent.unique()),['#087E8B','#7856A6','#D08B36','#527D9A','#C05D7C','#617B58'][:pop.Continent.nunique()]))
fig = px.treemap(pop, path=[px.Constant('World'),'Continent','Country'],
                 values='2022 Population',color='Continent',
                 color_discrete_map=continent_colors,template='plotly_white',
                 title='Asia contains nearly three-fifths of the world’s population')
fig.update_traces(maxdepth=3, textfont=dict(size=12), texttemplate='%{label}<br>%{percentRoot:.1%}',
                  hovertemplate='<b>%{label}</b><br>Population: %{value:,.0f}<br>World share: %{percentRoot:.2%}<extra></extra>',
                  marker=dict(line=dict(width=1,color='white')))
fig.update_layout(height=650,title_x=.02,margin=dict(l=15,r=15,t=85,b=15))



continent_totals=pop.groupby('Continent')['2022 Population'].sum().sort_values(ascending=False)
asia_share=continent_totals.loc['Asia']/continent_totals.sum()
fig.add_annotation(x=.01,y=1.10,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>Asia holds {asia_share:.1%} of world population; larger tiles mean more people.</b>',
                   font=dict(size=14,color='#24364B'))
fig.update_layout(margin=dict(l=15,r=15,t=130,b=15))
fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** A 100% stacked bar chart compares part-to-whole composition across years. Africa is highlighted in orange, other groups provide muted context, and the same order is retained for every year so the changing shares are easy to follow (Lecture 7).

**What the chart shows:** Africa's share of world population increased from about 9.9% in 1970 to 17.9% in 2022, while Europe's share fell from about 17.8% to 9.3%. Asia stayed the largest group throughout. I used stacked bars because each bar represents the whole world in that year, so the changing proportions are easy to compare. Keeping the colours and stacking order consistent makes the trend easier to read.


In [8]:
# The key finding is shown through the title, labels and annotations.
# Q7 — Lecture 7: 100% stacked composition, Africa accent
shares = df_long.copy()
shares['Group'] = shares['Continent'].where(shares['Continent'].isin(['Africa','Asia','Europe']),'Rest of the world')
shares = shares.groupby(['Year','Group'],as_index=False)['Population'].sum()
shares['Share'] = shares['Population']/shares.groupby('Year')['Population'].transform('sum')
shares['Label'] = np.where(shares.Group.eq('Africa'),shares.Share.map('{:.1%}'.format),'')
order=['Africa','Asia','Europe','Rest of the world']
fig = px.bar(shares,x='Year',y='Share',color='Group',text='Label',
             category_orders={'Group':order},
             color_discrete_map={'Africa':'#7856A6','Asia':'#527D9A','Europe':'#B5C4D1','Rest of the world':'#E4EAF0'},
             template='plotly_white',
             title='Africa’s share of world population rose while Europe’s fell',
             labels={'Share':'Share of world population'})
fig.update_traces(textposition='inside',textfont=dict(size=11))
fig.update_layout(barmode='stack',height=530,title_x=.02,legend_title_text='',
                  legend=dict(orientation='h',y=-.2,x=0),
                  xaxis_title='',yaxis_title='World population share',
                  margin=dict(l=70,r=35,t=85,b=110),bargap=.22)
fig.update_xaxes(type='category',showgrid=False)
fig.update_yaxes(tickformat='.0%',range=[0,1],showgrid=False)



# Label all four shares in the first and last bars; show the magnitude of change explicitly.
share_table=shares.pivot(index='Year',columns='Group',values='Share')
a0,a1=share_table.loc[1970,'Africa'],share_table.loc[2022,'Africa']
e0,e1=share_table.loc[1970,'Europe'],share_table.loc[2022,'Europe']
fig.add_annotation(x=.01,y=1.13,xref='paper',yref='paper',showarrow=False,xanchor='left',
                   text=f'<b>Africa {a0:.1%} → {a1:.1%} (+{(a1-a0)*100:.1f} pp); Europe {e0:.1%} → {e1:.1%} ({(e1-e0)*100:.1f} pp).</b>',
                   font=dict(size=13,color='#24364B'))
fig.update_layout(margin=dict(l=70,r=35,t=140,b=110))
fig.show()
